# TV1 — Console Dataset ETL

This notebook preprocesses only the VGChartz Console dataset and exports `data/processed/vgchartz_cleaned.csv`.

> **Input-location note:** the project documentation specifies `data/raw/vgchartz_games.csv`, but the inspected repository currently stores the file at its root. The load cell checks the documented path first, then safely falls back to the root-level file.

## 1. Load and Inspect Data

Load the source, confirm its schema, and retain all original columns.

In [58]:
from pathlib import Path
import pandas as pd
import os


documented_input = os.path.join('..', '..', 'data', 'raw', 'vgchartz_games.csv')
fallback_input = Path("vgchartz_games.csv")


df = pd.read_csv(documented_input)
rows_before = len(df)
print(f"Loaded {rows_before:,} rows from: {documented_input}")
display(df.head())
print(df.dtypes)
print("Columns:", df.columns.tolist())

Loaded 4,000 rows from: ..\..\data\raw\vgchartz_games.csv


,Pos,Game,Platform,Genre,Publisher,Developer,Critic Score,User Score,Metacritic (RAWG),VGChartz Score,Total Shipped,Total Sales,NA Sales,PAL Sales,Japan Sales,Other Sales,Release Date,Release Year,Last Update
0,1,Tetris,Series,Puzzle,The Tetris Company,Alexey Pajitnov,NaN,8.7,NaN,NaN,520.00m,NaN,NaN,NaN,NaN,NaN,01st Jan 88,1988.0,27th Feb 20
1,2,Pokemon,Series,Misc,Nintendo,Game Freak,NaN,NaN,NaN,NaN,516.33m,NaN,NaN,NaN,NaN,NaN,28th Sep 98,1998.0,03rd Feb 20
2,3,Call of Duty,Series,Shooter,Activision,Infinity Ward,9.1,8.3,91.0,NaN,500.00m,NaN,NaN,NaN,NaN,NaN,29th Oct 03,2003.0,03rd Feb 20
3,4,Grand Theft Auto,Series,Action,Rockstar Games,Rockstar North,NaN,7.7,NaN,NaN,475.00m,NaN,NaN,NaN,NaN,NaN,27th Mar 98,1998.0,03rd Feb 20
4,5,Super Mario,Series,Platformer,Nintendo,Nintendo,NaN,7.4,NaN,NaN,434.57m,NaN,NaN,NaN,NaN,NaN,20th Jul 83,1983.0,20th Feb 20


Pos                    int64
Game                     str
Platform                 str
Genre                    str
Publisher                str
Developer                str
Critic Score         float64
User Score           float64
Metacritic (RAWG)    float64
VGChartz Score       float64
Total Shipped            str
Total Sales          float64
NA Sales             float64
PAL Sales            float64
Japan Sales          float64
Other Sales          float64
Release Date             str
Release Year         float64
Last Update              str
dtype: object
Columns: ['Pos', 'Game', 'Platform', 'Genre', 'Publisher', 'Developer', 'Critic Score', 'User Score', 'Metacritic (RAWG)', 'VGChartz Score', 'Total Shipped', 'Total Sales', 'NA Sales', 'PAL Sales', 'Japan Sales', 'Other Sales', 'Release Date', 'Release Year', 'Last Update']


## 2. Process Total Shipped

`Total Shipped` is stored as text in millions (for example, `520.00m`). Invalid or unavailable values are retained as missing after numeric conversion.

In [59]:
print(df["Total Shipped"].value_counts(dropna=False).head(10))

shipped_millions = pd.to_numeric(
    df["Total Shipped"].astype("string").str.strip().str.replace(r"m$", "", regex=True),
    errors="coerce",
)
df["Volume_Units"] = shipped_millions * 1_000_000
df[["Total Shipped", "Volume_Units"]].head()

Total Shipped
1.00m    278
0.50m    123
2.00m    108
0.20m     91
0.15m     86
3.00m     82
0.25m     71
1.50m     69
5.00m     65
0.16m     64
Name: count, dtype: int64


,Total Shipped,Volume_Units
0,520.00m,520000000.0
1,516.33m,516330000.0
2,500.00m,500000000.0
3,475.00m,475000000.0
4,434.57m,434570000.0


## 3. Normalize Release Date

The source uses ordinal day suffixes and two-digit years (for example, `01st Jan 88`). Invalid or missing dates become missing values; no dates are invented.

In [60]:
release_text = df["Release Date"].astype("string").str.strip()
release_text = release_text.str.replace(r"^(\d{1,2})(?:st|nd|rd|th)", r"\1", regex=True)
release_dates = pd.to_datetime(release_text, format="%d %b %y", errors="coerce")

df["Release Date"] = release_dates.dt.strftime("%Y-%m-%d")
df["Release_Year"] = release_dates.dt.year.astype("Int64")
df[["Release Date", "Release_Year"]].head()

,Release Date,Release_Year
0,1988-01-01,1988
1,1998-09-28,1998
2,2003-10-29,2003
3,1998-03-27,1998
4,1983-07-20,1983


## 4. Calculate MSRP and Harmonized Revenue

`MSRP_USD` is a project-defined, standardized estimate: $50 through 2005, $60 for 2006–2020, and $70 from 2021. `Harmonized_Revenue_USD` is therefore not actual historical revenue; it is a comparable estimate based on shipped volume and the assigned MSRP.

In [61]:
df["MSRP_USD"] = pd.Series(pd.NA, index=df.index, dtype="Float64")
df.loc[df["Release_Year"] <= 2005, "MSRP_USD"] = 50.0
df.loc[df["Release_Year"].between(2006, 2020), "MSRP_USD"] = 60.0
df.loc[df["Release_Year"] >= 2021, "MSRP_USD"] = 70.0
df["Harmonized_Revenue_USD"] = df["Volume_Units"] * df["MSRP_USD"]
df[["Release_Year", "Volume_Units", "MSRP_USD", "Harmonized_Revenue_USD"]].head()

,Release_Year,Volume_Units,MSRP_USD,Harmonized_Revenue_USD
0,1988,520000000.0,50.0,26000000000.0
1,1998,516330000.0,50.0,25816500000.000004
2,2003,500000000.0,50.0,25000000000.0
3,1998,475000000.0,50.0,23750000000.0
4,1983,434570000.0,50.0,21728500000.0


## 5. Process Critic Score

The inspected `Critic Score` values are on a 0–10 scale and use `N/A` for missing data. Missing scores are imputed from the median valid score of the exact corresponding `Genre`; genres with no valid score remain missing. The original column is preserved.

Standardized_Rating này không thực sự cần thiết nếu sau này bạn muốn so sánh:

PC vs Console vs Mobile

Bởi vì:

Critic Score của VGChartz là một loại điểm đánh giá.
Steam có thể có positive/negative reviews.
Mobile có star rating.

Ba loại điểm này không phải cùng một thước đo.
Console: Critic Score = 80
Steam: Positive Ratio = 80%
Mobile: Rating = 4.0/5

Không thể nói đơn giản rằng cả ba đều tương đương với "80/100".

Vì vậy, trong project của bạn, mình sẽ không khuyến nghị dùng Standardized_Rating làm biến chính để so sánh PC/Console/Mobile.

In [62]:
critic_numeric = pd.to_numeric(df["Critic Score"], errors="coerce")
print(critic_numeric.describe())

genre_median_critic = critic_numeric.groupby(df["Genre"]).transform("median")
critic_imputed = critic_numeric.fillna(genre_median_critic)
df["Rating"] = critic_imputed * 10
df[["Genre", "Critic Score", "Rating"]].head(10)

def get_tier(score):
  if score >= 85.0:
    return 'Masterpiece'
  elif score >= 75.0:
    return 'Great'
  elif score >= 50.0:
    return 'Mixed'
  else:
    return 'Poor'

df['Rating_Tier'] = df['Rating'].apply(get_tier)

count    2470.000000
mean        7.897611
std         1.033152
min         3.000000
25%         7.300000
50%         8.100000
75%         8.600000
max        10.000000
Name: Critic Score, dtype: float64


## 6. Create Console_Maker and Console_Gen

Mappings use the actual `Platform` codes. Only platforms explicitly named by the task receive a console-generation label; unsupported, non-console, aggregate, and older platforms remain missing for `Console_Gen` and are `Other` for `Console_Maker`.

In [63]:
print(sorted(df["Platform"].dropna().unique()))

sony_platforms = {"PS", "PS2", "PS3", "PS4", "PS5", "PSP", "PSV", "PSN"}
microsoft_platforms = {"XB", "X360", "XOne", "XS", "XBL"}
nintendo_platforms = {"NES", "SNES", "N64", "GC", "Wii", "WiiU", "NS", "NS2", "GB", "GBC", "GBA", "DS", "3DS"}

def console_maker(platform):
    if platform in sony_platforms:
        return "Sony"
    if platform in microsoft_platforms:
        return "Microsoft"
    if platform in nintendo_platforms:
        return "Nintendo"
    return "Other"

generation_map = {
    "PS2": "Gen 6 (PS2/Xbox/GC)", "XB": "Gen 6 (PS2/Xbox/GC)", "GC": "Gen 6 (PS2/Xbox/GC)",
    "PS3": "Gen 7 (PS3/X360/Wii)", "X360": "Gen 7 (PS3/X360/Wii)", "Wii": "Gen 7 (PS3/X360/Wii)",
    "PS4": "Gen 8 (PS4/XOne/Switch/WiiU)", "XOne": "Gen 8 (PS4/XOne/Switch/WiiU)", "NS": "Gen 8 (PS4/XOne/Switch/WiiU)", "WiiU": "Gen 8 (PS4/XOne/Switch/WiiU)",
    "PS5": "Gen 9 (PS5/XSX)", "XS": "Gen 9 (PS5/XSX)",
}

df["Console_Maker"] = df["Platform"].map(console_maker)
df["Console_Gen"] = df["Platform"].map(generation_map).astype("string")
df[["Platform", "Console_Maker", "Console_Gen"]].head(10)

['2600', '3DO', '3DS', 'All', 'Amig', 'DC', 'DS', 'GB', 'GBA', 'GBC', 'GC', 'GEN', 'Int', 'N64', 'NES', 'NS', 'NS2', 'PC', 'PS', 'PS2', 'PS3', 'PS4', 'PS5', 'PSN', 'PSP', 'PSV', 'SAT', 'SCD', 'SNES', 'Series', 'Wii', 'WiiU', 'X360', 'XB', 'XBL', 'XOne', 'XS', 'iOS']


,Platform,Console_Maker,Console_Gen
0,Series,Other,<NA>
1,Series,Other,<NA>
2,Series,Other,<NA>
3,Series,Other,<NA>
4,Series,Other,<NA>
5,Series,Other,<NA>
6,All,Other,<NA>
7,Series,Other,<NA>
8,Series,Other,<NA>
9,All,Other,<NA>


## 7. Basic Validation

Confirm row preservation, important missing values, new-column types, representative rows, and mapping counts.

In [64]:
print(f"Rows before: {rows_before:,} | Rows after: {len(df):,}")
assert len(df) == rows_before, "Row count changed unexpectedly."

new_columns = ["Volume_Units", "Release_Year", "MSRP_USD", "Harmonized_Revenue_USD", "Rating", "Rating_Tier", "Console_Maker", "Console_Gen"]
print("\nMissing values:")
print(df[["Total Shipped", "Release Date", "Critic Score"] + new_columns].isna().sum())
print("\nNew column data types:")
print(df[new_columns].dtypes)
print("\nConsole maker counts:")
print(df["Console_Maker"].value_counts(dropna=False))
print("\nConsole generation counts:")
print(df["Console_Gen"].value_counts(dropna=False))
display(df[["Game", "Platform", "Total Shipped", "Volume_Units", "Release Date", "Release_Year", "Rating", "Rating_Tier", "Console_Maker", "Console_Gen"]].head())

Rows before: 4,000 | Rows after: 4,000

Missing values:
Total Shipped                0
Release Date               417
Critic Score              1530
Volume_Units                 0
Release_Year               417
MSRP_USD                   417
Harmonized_Revenue_USD     417
Rating                     186
Rating_Tier                  0
Console_Maker                0
Console_Gen               3413
dtype: int64

New column data types:
Volume_Units              Float64
Release_Year                Int64
MSRP_USD                  Float64
Harmonized_Revenue_USD    Float64
Rating                    float64
Rating_Tier                   str
Console_Maker                 str
Console_Gen                string
dtype: object

Console maker counts:
Console_Maker
Other        2844
Nintendo      672
Sony          396
Microsoft      88
Name: count, dtype: int64

Console generation counts:
Console_Gen
<NA>                            3413
Gen 8 (PS4/XOne/Switch/WiiU)     226
Gen 7 (PS3/X360/Wii)           

,Game,Platform,Total Shipped,Volume_Units,Release Date,Release_Year,Rating,Rating_Tier,Console_Maker,Console_Gen
0,Tetris,Series,520.00m,520000000.0,1988-01-01,1988,80.0,Great,Other,<NA>
1,Pokemon,Series,516.33m,516330000.0,1998-09-28,1998,77.0,Great,Other,<NA>
2,Call of Duty,Series,500.00m,500000000.0,2003-10-29,2003,91.0,Masterpiece,Other,<NA>
3,Grand Theft Auto,Series,475.00m,475000000.0,1998-03-27,1998,83.0,Great,Other,<NA>
4,Super Mario,Series,434.57m,434570000.0,1983-07-20,1983,86.0,Masterpiece,Other,<NA>


## 8. Export Cleaned Dataset

Create the requested processed-data directory when needed and write the complete cleaned Console dataset.

In [65]:
output_path = os.path.join('..', '..', 'data', 'processed', 'vgchartz_cleaned.csv')
df.to_csv(output_path, index=False)
print(f"Exported {len(df):,} rows to: {output_path}")

Exported 4,000 rows to: ..\..\data\processed\vgchartz_cleaned.csv
